# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ismayil282/flyrank-ml-engineering-task-1/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Lane 2 — Refresh / Content Opportunity Scoring.**

FlyRank's reviewers can only manually check a limited number of pages each week. Right now that
triage is either done by gut feeling or by the product's own rule-based flags. My question is
whether a transparent, evidence-based ranking can point reviewers at the pages most worth their
time — pages with real search demand that are declining, under-clicking, or going stale — instead
of a flat list or a black-box score.

I'm picking this lane over Lane 1 (signal analysis) and Lane 4 (CTR-only) because the starter data
already shows a large, non-trivial slice of pages sitting in "worth reviewing" territory (numbers
below), and because the starter pipeline gives me a working baseline-to-model comparison
(Precision@50 rising from ~0.24 to ~0.74) to try to reproduce and then improve on with the full
warehouse. Lane 3 (clustering) is interesting but a weaker fit for a "what should someone do
Monday morning" deliverable, which is the action I actually want to support.

In [1]:
# No computation needed for this section — see Section 3 for the supporting numbers.


## 2. The question: decision, action, cost of a wrong call

**Search question:** Given a client's content inventory and its last 90 days of search and
engagement signals, which pages should a content reviewer look at first, and why?

**Unit of analysis:** one page (`content_id`), scoped within a client (`client_id`) — not a
client, not a query, not a day. A page is the thing someone actually edits.

**Output:** a ranked queue of pages, each with a score and a reason code (e.g. "declining with
demand," "stale but visible," "under-clicking for its position").

**The decision / the action:** a content or SEO reviewer decides which pages to manually review
and refresh this week, out of far more candidates than they have time for. The ranked list is
the thing they open Monday morning; the action is "open this page, read the reason code, decide
whether to rewrite, expand, or leave it."

**Cost of a wrong recommendation:** if the model sends a reviewer to a page that isn't actually a
problem (a false positive), the cost is wasted reviewer time — a real but bounded cost, since a
human still makes the final call before touching anything. If the model misses a genuinely
declining, high-demand page (a false negative), the cost is a page that keeps losing visibility
and traffic unnoticed for another cycle — a compounding, harder-to-see cost. Because these costs
are asymmetric and reviewer time is the real constraint, precision in the top of the ranked list
(e.g. Precision@50) matters more here than trying to catch every possible case.

**Why data or ML can help at all:** with a handful of pages, a person can just look. With
thousands of pages per client and dozens of clients, nobody can manually scan everything every
week — the bottleneck is attention, not access to the data. A model doesn't decide anything on
its own; it just orders the queue so limited human attention goes to the pages with the
strongest evidence behind them first.

In [2]:
# No computation needed for this section — the framing above is conceptual.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import os, sys, subprocess
import pandas as pd

# Works whether run in Colab, locally from work/notebooks/, or from the repo root.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ismayil282/flyrank-ml-engineering-task-1"
REPO_DIR = "flyrank-ml-engineering-task-1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")  # move from work/notebooks/ to the repo root

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

n_rows = len(df)
n_clients = df["client_id"].nunique()

# How many pages are "declining with demand" -- a real starter reason code for this lane
declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).sum()

# How many visible pages are under-capturing clicks for their position
low_ctr_visible = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0) & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
).sum()

print(f"rows: {n_rows} | clients: {n_clients}")
print(f"declining_with_demand: {declining_with_demand} pages "
      f"({declining_with_demand / n_rows:.1%} of the dataset)")
print(f"low_ctr_visible_page: {low_ctr_visible} pages "
      f"({low_ctr_visible / n_rows:.1%} of the dataset)")
print(f"overall trend_direction == 'down' share: {(df['trend_direction']=='down').mean():.1%}")


rows: 30000 | clients: 32
declining_with_demand: 13152 pages (43.8% of the dataset)
low_ctr_visible_page: 9759 pages (32.5% of the dataset)
overall trend_direction == 'down' share: 54.2%


## 4. Careful words: what I can and can't claim

**What I can claim:** which pages, in this snapshot, show observed search and engagement signals
consistent with decline, staleness, or CTR under-performance for their position — and a ranked,
evidence-backed list of which ones look worth reviewing first, with reason codes attached. I can
also claim, on the starter data, that a learned ranking outperforms a simple hand-written rule at
this task (Precision@50 rising from about 0.24 to 0.74 in the starter pipeline) — a result I'll
need to re-earn on my own chosen window and validation split, not just repeat.

**What I can't claim:** that refreshing a page will cause it to recover — that needs a real
experiment or a causal design, which this data alone can't give me. I also can't claim to have
detected a Google ranking factor, predicted "the algorithm," or found true semantic meaning in the
content (the data has no article text, only metrics and metadata). Every result here is
observational: I watched what happened, I didn't intervene. My output supports a human reviewer's
decision — it doesn't replace their judgment, and a high score is a candidate to check, not a
verdict.

In [4]:
# No computation needed for this section — the caveats above are qualitative.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.